In [1]:
# Logic taken from Madhur's repo on knowledge capacity
import random
import re

import numpy as np
from transformers import AutoTokenizer
from datasets import load_dataset

/nlpscratch/home/ralvarez/memorization-llms/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_wikitext(
    min_length: int = 100,
    max_length: int = 300,
    num_proc: int = 20,
    save_path: str = "wikitext_processed",
):
    """
    Load and process wikitext dataset, tokenize original and shuffled text, and save everything.

    Args:
        min_length: Minimum token length
        max_length: Maximum token length
        num_proc: Number of processes for parallel processing
        save_path: Path to save the dataset directory

    Saves:
        Dataset directory with columns:
            - orig_index: Original WikiText indices
            - text: Original text strings
            - input_ids: Tokenized original text
            - attention_mask: Attention masks for original text
            - shuffled_text: Word-shuffled text strings
            - shuffled_input_ids: Tokenized shuffled text
            - shuffled_attention_mask: Attention masks for shuffled text
    """
    # Load dataset
    dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")

    # Add original indices
    dataset = dataset.map(
        lambda example, idx: {"orig_index": idx},
        with_indices=True,
        num_proc=num_proc,
        desc="Adding indices",
    )

    # Remove headers/titles (lines wrapped in equals signs)
    pattern = r"(?: =)+[^=]+(?:= )+"
    dataset = dataset.filter(
        lambda x: not re.match(pattern, x["text"]),
        num_proc=num_proc,
        desc="Filtering headers",
    )

    # Tokenize original text
    tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token

    def tokenize(examples):
        return tokenizer(
            examples["text"],
            padding="max_length",
            max_length=max_length,
        )

    dataset = dataset.map(
        tokenize, batched=True, num_proc=num_proc, desc="Tokenizing original text"
    )

    def filter_length(batch):
        return [
            min_length <= np.sum(mask) <= max_length for mask in batch["attention_mask"]
        ]

    dataset = dataset.filter(
        filter_length,
        batched=True,
        num_proc=num_proc,
        desc="Filtering by length of original text",
    )

    def shuffle_words_batch(examples):
        """Shuffle words in each text sequence."""
        shuffled = []
        for text in examples["text"]:
            words = text.split()
            random.shuffle(words)
            shuffled.append(" ".join(words))
        return {"shuffled_text": shuffled}

    dataset = dataset.map(
        shuffle_words_batch, batched=True, num_proc=num_proc, desc="Shuffling words"
    )

    # Tokenize shuffled text
    def tokenize_shuffled(examples):
        result = tokenizer(
            examples["shuffled_text"],
            padding="max_length",
            max_length=max_length,
        )
        return {
            "shuffled_input_ids": result["input_ids"],
            "shuffled_attention_mask": result["attention_mask"],
        }

    dataset = dataset.map(
        tokenize_shuffled,
        batched=True,
        num_proc=num_proc,
        desc="Tokenizing shuffled text",
    )

    print(f"Dataset after processing has {len(dataset)} sequences.")

    # Filter shuffled text by length
    def filter_shuffled_length(batch):
        return [
            min_length <= np.sum(mask) <= max_length
            for mask in batch["shuffled_attention_mask"]
        ]

    dataset = dataset.filter(
        filter_shuffled_length,
        batched=True,
        num_proc=num_proc,
        desc="Filtering shuffled by length",
    )

    print(f"Dataset after filtering shuffled has {len(dataset)} sequences.")

    # Save dataset to disk
    print(f"Saving processed dataset to {save_path}...")
    dataset.save_to_disk(save_path)

    return dataset

In [3]:
process_wikitext()

Dataset after processing has 493311 sequences.


Filtering shuffled by length (num_proc=20): 100%|██████████| 493311/493311 [00:06<00:00, 81763.35 examples/s] 


Dataset after filtering shuffled has 486299 sequences.
Saving processed dataset to wikitext_processed...


Saving the dataset (8/8 shards): 100%|██████████| 486299/486299 [00:31<00:00, 15497.91 examples/s]


Dataset({
    features: ['text', 'orig_index', 'input_ids', 'attention_mask', 'shuffled_text', 'shuffled_input_ids', 'shuffled_attention_mask'],
    num_rows: 486299
})

In [4]:
# Example: Load processed dataset
from datasets import load_from_disk

dataset = load_from_disk("wikitext_processed")
print(f"Loaded {len(dataset)} sequences")
print(f"Columns: {dataset.column_names}")

Loaded 486299 sequences
Columns: ['text', 'orig_index', 'input_ids', 'attention_mask', 'shuffled_text', 'shuffled_input_ids', 'shuffled_attention_mask']
